# Lab 8 - Text Clustering

CISB5123 Text Analytics

TEXT CLUSTERING USING TF-IDF VECTORIZER 

In [ ]:
# Step 1: Import the libraries
import numpy as np
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from tabulate import tabulate
from collections import Counter

# Step 2: Create the documents
dataset = ["I love playing football on the weekends",
 "I enjoy hiking and camping in the mountains",
 "I like to read books and watch movies",
 "I prefer playing video games over sports",
 "I love listening to music and going to concerts"]

# Step 3: Vectorize the dataset
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(dataset)

# Step 4: Perform clustering
k = 2 # Define the number of clusters
km = KMeans(n_clusters=k)
km.fit(X)
# Predict the clusters for each document
y_pred = km.predict(X)
# Display the document and its predicted cluster in a table
table_data = [["Document", "Predicted Cluster"]]
table_data.extend([[doc, cluster] for doc, cluster in zip(dataset, y_pred)])
print(tabulate(table_data, headers="firstrow"))
# Print top terms per cluster
print("\nTop terms per cluster:")
order_centroids = km.cluster_centers_.argsort()[:, ::-1]
terms = vectorizer.get_feature_names_out()
for i in range(k):
 print("Cluster %d:" % i)
 for ind in order_centroids[i, :10]:
    print(' %s' % terms[ind])
 print()

# Step 5: Evaluate results
# Calculate purity
total_samples = len(y_pred)
cluster_label_counts = [Counter(y_pred)]
purity = sum(max(cluster.values()) for cluster in cluster_label_counts) / total_samples
print("Purity:", purity)

Document                                           Predicted Cluster
-----------------------------------------------  -------------------
I love playing football on the weekends                            1
I enjoy hiking and camping in the mountains                        1
I like to read books and watch movies                              0
I prefer playing video games over sports                           1
I love listening to music and going to concerts                    0

Top terms per cluster:
Cluster 0:
 to
 and
 read
 like
 books
 movies
 watch
 music
 listening
 going

Cluster 1:
 playing
 the
 weekends
 on
 football
 prefer
 video
 over
 sports
 games

Purity: 0.6


TEXT CLUSTERING USING WORD2VEC VECTORIZER

In [10]:
# Step 1: Import the libraries
import numpy as np
from sklearn.cluster import KMeans
from gensim.models import Word2Vec
from tabulate import tabulate
from collections import Counter

# Step 2: Create the documents
dataset = ["I love playing football on the weekends",
 "I enjoy hiking and camping in the mountains",
 "I like to read books and watch movies",
 "I prefer playing video games over sports",
 "I love listening to music and going to concerts"]

# Step 3: Train Word2Vec model
tokenized_dataset = [doc.split() for doc in dataset]
word2vec_model = Word2Vec(sentences=tokenized_dataset, vector_size=100,
window=5, min_count=1, workers=4)

# Step 4: Create document embeddings
X = np.array([np.mean([word2vec_model.wv[word] for word in doc.split() if word in
word2vec_model.wv], axis=0) for doc in dataset])

# Step 5: Perform clustering
k = 2 # Define the number of clusters
km = KMeans(n_clusters=k)
km.fit(X)
# Predict the clusters for each document
y_pred = km.predict(X)
# Tabulate the document and predicted cluster
table_data = [["Document", "Predicted Cluster"]]
table_data.extend([[doc, cluster] for doc, cluster in zip(dataset, y_pred)])
print(tabulate(table_data, headers="firstrow"))

#Step 6: Evaluate results
# Calculate purity
total_samples = len(y_pred)
cluster_label_counts = [Counter(y_pred)]
purity = sum(max(cluster.values()) for cluster in cluster_label_counts) / total_samples
print("Purity:", purity)

Document                                           Predicted Cluster
-----------------------------------------------  -------------------
I love playing football on the weekends                            1
I enjoy hiking and camping in the mountains                        0
I like to read books and watch movies                              1
I prefer playing video games over sports                           0
I love listening to music and going to concerts                    1
Purity: 0.6


# EXERCISE

1. Modify the codes for both TF-IDF & Word2Vec vectorizer by adding text preprocessing steps. 
2. Perform text clustering on 'customer_complaints_1.csv' dataset, specifically the Text column.

In [11]:
import re
import numpy as np
import pandas as pd

from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics import accuracy_score
from collections import Counter
from gensim.models import Word2Vec

# Load dataset
df = pd.read_csv("customer_complaints_1.csv")
texts = df["text"].fillna("").astype(str).tolist()

# Use rating as the reference label for purity
true_labels = df["rating"].tolist()

# Text preprocessing function
stop_words = set(ENGLISH_STOP_WORDS)

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+|[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    tokens = [word for word in text.split() if word not in stop_words and len(word) > 2]
    return " ".join(tokens)

def tokenize_preprocessed(text):
    return preprocess_text(text).split()

# Purity function
def purity_score(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    total = 0

    for cluster_id in np.unique(y_pred):
        idx = (y_pred == cluster_id)
        cluster_labels = y_true[idx]
        if len(cluster_labels) > 0:
            total += Counter(cluster_labels).most_common(1)[0][1]

    return total / len(y_true)

# Helper to run KMeans
def run_kmeans(X, k=2):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    y_pred = km.fit_predict(X)
    return km, y_pred

# A) TF-IDF WITHOUT PREPROCESSING
tfidf_raw = TfidfVectorizer()
X_tfidf_raw = tfidf_raw.fit_transform(texts)
km_tfidf_raw, pred_tfidf_raw = run_kmeans(X_tfidf_raw, k=2)
purity_tfidf_raw = purity_score(true_labels, pred_tfidf_raw)

# B) TF-IDF WITH PREPROCESSING
clean_texts = [preprocess_text(t) for t in texts]
tfidf_clean = TfidfVectorizer()
X_tfidf_clean = tfidf_clean.fit_transform(clean_texts)
km_tfidf_clean, pred_tfidf_clean = run_kmeans(X_tfidf_clean, k=2)
purity_tfidf_clean = purity_score(true_labels, pred_tfidf_clean)

# C) WORD2VEC WITHOUT PREPROCESSING
raw_tokens = [re.findall(r"\b\w+\b", t.lower()) for t in texts]
w2v_raw = Word2Vec(
    sentences=raw_tokens,
    vector_size=100,
    window=5,
    min_count=1,
    workers=1,
    seed=42
)

def doc_vector(tokens, model):
    vectors = [model.wv[w] for w in tokens if w in model.wv]
    if len(vectors) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

X_w2v_raw = np.vstack([doc_vector(tokens, w2v_raw) for tokens in raw_tokens])
km_w2v_raw, pred_w2v_raw = run_kmeans(X_w2v_raw, k=2)
purity_w2v_raw = purity_score(true_labels, pred_w2v_raw)

# D) WORD2VEC WITH PREPROCESSING
clean_tokens = [tokenize_preprocessed(t) for t in texts]
w2v_clean = Word2Vec(
    sentences=clean_tokens,
    vector_size=100,
    window=5,
    min_count=1,
    workers=1,
    seed=42
)

X_w2v_clean = np.vstack([doc_vector(tokens, w2v_clean) for tokens in clean_tokens])
km_w2v_clean, pred_w2v_clean = run_kmeans(X_w2v_clean, k=2)
purity_w2v_clean = purity_score(true_labels, pred_w2v_clean)

# Show results
results = pd.DataFrame({
    "Method": [
        "TF-IDF (raw)",
        "TF-IDF (preprocessed)",
        "Word2Vec (raw)",
        "Word2Vec (preprocessed)"
    ],
    "Purity": [
        purity_tfidf_raw,
        purity_tfidf_clean,
        purity_w2v_raw,
        purity_w2v_clean
    ]
})

print(results)

print("\nTF-IDF (preprocessed) cluster assignments:")
for text, cluster in zip(texts, pred_tfidf_clean):
    print(cluster, "->", text[:80], "...")

print("\nWord2Vec (preprocessed) cluster assignments:")
for text, cluster in zip(texts, pred_w2v_clean):
    print(cluster, "->", text[:80], "...")

                    Method    Purity
0             TF-IDF (raw)  0.947368
1    TF-IDF (preprocessed)  0.947368
2           Word2Vec (raw)  0.947368
3  Word2Vec (preprocessed)  0.947368

TF-IDF (preprocessed) cluster assignments:
1 -> I used to love Comcast. Until all these constant updates. My internet and cable  ...
1 -> I'm so over Comcast! The worst internet provider. I'm taking online classes and  ...
1 -> If I could give them a negative star or no stars on this review I would. I have  ...
1 -> I've had the worst experiences so far since install on 10/4/16. Nothing but prob ...
1 -> Check your contract when you sign up for Comcast as their advertised offers do n ...
1 -> Thank God. I am changing to Dish. They gave me awesome pricing and super people  ...
1 -> I Have been a long time customer and only have Xfinity as my ISP for a while now ...
1 -> There is a malfunction on the DVR manager which is preventing us from adding mor ...
0 -> Charges overwhelming. Comcast service rep was 

Do the Purity differ when applying text preprocessing before vectorization?
No, the purity does not change in this small dataset.

In [12]:
import re
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from collections import Counter
import numpy as np

df = pd.read_csv("customer_complaints_1.csv")
texts = df["text"].fillna("").astype(str).tolist()
labels = df["rating"].tolist()

stop_words = set(ENGLISH_STOP_WORDS)

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+|[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    tokens = [w for w in text.split() if w not in stop_words and len(w) > 2]
    return " ".join(tokens)

def purity_score(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    total = 0
    for cluster_id in np.unique(y_pred):
        idx = (y_pred == cluster_id)
        total += Counter(y_true[idx]).most_common(1)[0][1]
    return total / len(y_true)

clean_texts = [preprocess_text(t) for t in texts]

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(clean_texts)

km = KMeans(n_clusters=2, random_state=42, n_init=10)
pred = km.fit_predict(X)

print("Purity:", purity_score(labels, pred))
for text, cluster in zip(texts, pred):
    print(cluster, "->", text[:80], "...")

Purity: 0.9473684210526315
1 -> I used to love Comcast. Until all these constant updates. My internet and cable  ...
1 -> I'm so over Comcast! The worst internet provider. I'm taking online classes and  ...
1 -> If I could give them a negative star or no stars on this review I would. I have  ...
1 -> I've had the worst experiences so far since install on 10/4/16. Nothing but prob ...
1 -> Check your contract when you sign up for Comcast as their advertised offers do n ...
1 -> Thank God. I am changing to Dish. They gave me awesome pricing and super people  ...
1 -> I Have been a long time customer and only have Xfinity as my ISP for a while now ...
1 -> There is a malfunction on the DVR manager which is preventing us from adding mor ...
0 -> Charges overwhelming. Comcast service rep was so ignorant and rude when I call t ...
1 -> I have had cable, DISH, and U-verse, etc. in the past. All are eh... but you kno ...
1 -> Had them from 2014 to now. I'd tell new customers to run but there i